In [ ]:
import requests
import pandas as pd
from datetime import datetime

def get_warframe_data(item_url_name):
    """
    item_url_name: 物品在网址里的名字，比如 'wisp_prime_set'
    """
    # 构造 API 地址
    api_url = f"https://api.warframe.market/v1/items/{item_url_name}/statistics"
    
    # 发送请求获取数据
    response = requests.get(api_url)
    
    if response.status_code == 200:
        data = response.json()
        
        # 3. 提取过去 90 天的数据 (取 daily 统计)
        # 要提取的是 'closed_min' (成交最低价), 'avg_price' (均价), 'datetime' (日期)
        rows = data['payload']['statistics_closed']['90days']
        
        # 4. 转换成 Pandas 的 DataFrame (数据表格格式)
        df = pd.DataFrame(rows)
        
        # 5. 时间格式转换（将原始字符串转成日期）
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # 只保留我们关心的几列：日期、均价、成交量、最低价
        df = df[['datetime', 'avg_price', 'volume', 'min_price']]
        
        return df
    else:
        print(f"报错啦！状态码是: {response.status_code}")
        return None

# --- 使用示例 ---
target_item = "wisp_prime_set" # 可以换成任何你想追踪的物品
df_wisp = get_warframe_data(target_item)

if df_wisp is not None:
    print(df_wisp.tail(10)) # 显示最后10行数据
    df_wisp.to_csv(f"{target_item}_history.csv", index=False)
    print(f"\n数据已保存为 {target_item}_history.csv")


                    datetime  avg_price  volume  min_price
79 2026-05-04 00:00:00+00:00       68.5     136         67
80 2026-05-05 00:00:00+00:00       88.5     256         50
81 2026-05-06 00:00:00+00:00      102.5     271         55
82 2026-05-07 00:00:00+00:00       75.0     254         50
83 2026-05-08 00:00:00+00:00       92.5     295         55
84 2026-05-09 00:00:00+00:00       75.0     295         50
85 2026-05-10 00:00:00+00:00      127.5     264         55
86 2026-05-11 00:00:00+00:00       68.5     206         67
87 2026-05-12 00:00:00+00:00       69.0     176         68
88 2026-05-13 00:00:00+00:00       69.0     180         68

数据已保存为 wisp_prime_set_history.csv
